### <font color= '#0400ffff'>**Index** </font><a class="anchor" id='toc'></a>

- [1. Introduction](#1)
- [2. Load libraries and install mandatory packages](#2)
- [3. Exploratory Data Analysis](#3)
    - [3.1. Import dataset](#3_1)
    - [3.2. Initial general analysis](#3_2)
        - [3.2.1. Data Types](#3_2_1)
        - [3.2.2. Check Missing Values](#3_2_2)
        - [3.2.3. Check for Duplicates](#3_2_3)
        - [3.2.4. Check unique values](#3_2_4)
- [4. Data Preparation](#4)


<a class="anchor" id="1">

# **1.  Introduction**

[Back to TOC](#toc)
</a>

**TO DO**: Introduzir uma intro do projeto 


<a class="anchor" id="2">

# **2.  Load libraries and install mandatory packages**

[Back to TOC](#toc)
</a>

In [1]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0"

In [2]:
# Install Java 17 (Required for Spark)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

Get:1 https://cli.github.com/packages stable InRelease [3917 B]
Get:2 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]       
Hit:3 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://archive.ubuntu.com/ubuntu noble InRelease [256 kB]               
Get:8 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]     
Get:9 https://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]       
Get:10 https://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]    
Get:11 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [64.3 kB]
Get:12 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [2412 kB]
Get:13 

In [37]:
import sqlite3
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, split, when, trim, size

# Set JAVA_HOME and initialize Spark Session with specific configurations
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


spark = SparkSession.builder \
        .master("local[4]") \
        .appName("PySpark DataFrames API") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

print("Spark Session configured and ready!")

Spark Session configured and ready!


<a class="anchor" id="3">

# **3.  Exploratory Data Analysis**

[Back to TOC](#toc)
</a>

<a class="anchor" id="3_1">

## **3.1. FIFA 21 Dataset - Metadata**

[Back to TOC](#toc)
</a>

#### Player Identity & Basics
* **ID**: The unique identification number assigned to each player in the database.
* **Name**: The short version of the player's name (e.g., L. Messi).
* **LongName**: The full name of the player.
* **photoUrl**: The web link to the player's profile picture.
* **playerUrl**: The web link to the player's full profile on SoFIFA.
* **Nationality**: The country the player is from.
* **Age**: The player's current age.
* **Club**: The team the player is currently playing for.
* **Contract**: The years remaining on the player's current contract or the contract type.
* **Positions**: All the positions the player is capable of playing.
* **Height**: The player's height (usually in cm).
* **Weight**: The player's weight (usually in kg).
* **Preferred Foot**: The foot (Left or Right) the player prefers to use.
* **Joined**: The exact date the player signed with his current club.
* **Loan Date End**: The date a player's loan period ends (if applicable).

#### Performance Ratings
* **OVA (Overall Rating)**: The general skill level of the player (0-99).
* **POT (Potential)**: The maximum rating the player can reach in his career.
* **BOV (Best Overall)**: The rating the player has in his best specific position.
* **Best Position**: The specific role where the player performs at his highest level.

#### Financial Information
* **Value**: The estimated market value of the player in Euros.
* **Wage**: The weekly salary the player earns at his club.
* **Release Clause**: The amount another club must pay to buy the player instantly.

#### Attacking Attributes
* **Attacking**: The total sum of points of all attacking categories.
* **Crossing**: Accuracy of crosses into the penalty area.
* **Finishing**: Ability to score goals when inside the box.
* **Heading Accuracy**: Accuracy of headers for passes or shots.
* **Short Passing**: Accuracy of short, ground-level passes.
* **Volleys**: Ability to strike the ball while it is in the air.

#### Skill Attributes
* **Skill**: The total sum of points of all technical skill categories.
* **Dribbling**: Ability to control the ball while moving at speed.
* **Curve**: Ability to bend the ball's trajectory (e.g., on free kicks).
* **FK Accuracy**: Accuracy when taking direct free kicks.
* **Long Passing**: Accuracy of long-distance or lobbed passes.
* **Ball Control**: Ability to control the ball instantly when receiving it.

#### Movement Attributes
* **Movement**: The total sum of points of all movement categories.
* **Acceleration**: How quickly a player reaches top speed.
* **Sprint Speed**: The maximum speed the player can reach.
* **Agility**: How quickly a player can change direction.
* **Reactions**: How fast a player responds to a situation happening around them.
* **Balance**: Ability to stay on his feet under physical pressure.

#### Power Attributes
* **Power**: The total sum of points of all physical power categories.
* **Shot Power**: How hard the player can kick the ball.
* **Jumping**: How high the player can jump for headers.
* **Stamina**: How long the player can run before getting tired.
* **Strength**: Physical power to push off opponents.
* **Long Shots**: Ability to score from outside the penalty area.

#### Mentality Attributes
* **Mentality**: The total sum of points of all mental categories.
* **Aggression**: The frequency and intensity of physical challenges and tackles.
* **Interceptions**: Ability to read the game and cut off opponent passes.
* **Positioning**: Ability to find open space on the field.
* **Vision**: Ability to see teammates' runs and execute difficult passes.
* **Penalties**: Accuracy and composure when taking penalty kicks.
* **Composure**: Ability to perform well under pressure from defenders.

#### Defending Attributes
* **Defending**: The total sum of points of all defensive categories.
* **Marking**: Ability to stay close to and track an opponent.
* **Standing Tackle**: Ability to take the ball away while standing.
* **Sliding Tackle**: Ability to take the ball away using a slide.

#### Goalkeeping Attributes
* **Goalkeeping**: The total sum of points of all goalkeeper skills.
* **GK Diving**: Ability to reach for the ball while jumping.
* **GK Handling**: Ability to catch and hold onto the ball.
* **GK Kicking**: Accuracy and distance of goal kicks.
* **GK Positioning**: Ability to stand in the right place to cover the goal.
* **GK Reflexes**: How fast the keeper reacts to close-range shots.

#### Playing Style & Summary Stats
* **Total Stats**: The sum of all individual attribute points of the player.
* **Base Stats**: The sum of the six main face-card attributes (Pace, Shooting, Passing, Dribbling, Defense, Physical).
* **W/F (Weak Foot)**: Performance with the non-dominant foot (1-5 stars).
* **SM (Skill Moves)**: Ability to perform dribbling tricks (1-5 stars).
* **A/W (Attacking Work Rate)**: Effort level in attack (Low/Medium/High).
* **D/W (Defensive Work Rate)**: Effort level in defense (Low/Medium/High).
* **IR (International Reputation)**: Global fame level (1-5 stars).
* **PAC (Pace)**: Overall speed rating.
* **SHO (Shooting)**: Overall scoring rating.
* **PAS (Passing)**: Overall playmaking rating.
* **DRI (Dribbling)**: Overall ball-carrying rating.
* **DEF (Defending)**: Overall defensive rating.
* **PHY (Physical)**: Overall physical presence rating.

#### Other
* **Hits**: Number of views or searches for the player on the platform.

<a class="anchor" id="3_1">

## **3.2.  Import dataset**

[Back to TOC](#toc)
</a>

In [4]:
df = spark.read.csv('./fifa21 raw data v2.csv', header=True, multiLine=True)

The .csv file contains random placed strings (") and "\n". Spark tries to read it but it counts as a new line in wrongfull places.

To correct it, when reading the .csv we added the parameter `multiline = True`.

In [5]:
# create a copy of the original dataframe to work with
fifa = df

In [6]:
#set employee_id as index
fifa = fifa.withColumn("ID", col("ID").cast("string"))

<a class="anchor" id="3_2">

## **3.3.  Inital general analysis**

[Back to TOC](#toc)
</a>

In [7]:
n_rows = fifa.count()
n_cols = len(fifa.columns)

print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")

Rows: 18979
Columns: 77


In [8]:
fifa.show(15)

26/05/14 10:22:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+-----------------+--------------------+--------------------+--------------------+-----------+---+----+---+--------------------+-----------+----------+------+------+--------------+---+-------------+------------+-------------+-------+-----+--------------+---------+--------+---------+----------------+-------------+-------+-----+---------+-----+-----------+------------+------------+--------+------------+------------+-------+---------+-------+-----+----------+-------+-------+--------+----------+---------+----------+-------------+-----------+------+---------+---------+---------+-------+---------------+--------------+-----------+---------+-----------+----------+--------------+-----------+-----------+----------+---+---+------+------+---+---+---+---+---+---+---+----+
|    ID|             Name|            LongName|            photoUrl|           playerUrl|Nationality|Age|↓OVA|POT|                Club|   Contract| Positions|Height|Weight|Preferred Foot|BOV|Best Position|      Joined|Loan 

By analysing some rows, we can already see some inconsistencies:

- `Club`: in the beggining of every cell in this column, there are a lot of "\n" which correspond to spaces which have no meaning in this case. We'll remove it.

- `Contract`: In this feature, there are two dates separated by a "~" for each contract which we assume that it corresponds to the beggining and end date of every player in said club. We will separate the column in two, one for the year of entering the club and other for termination of contract.

- `Positions`: It would be interesting to have a new column based on this one to count the number of positions each player has since some of them has more than one position.

- `Height and Weight`: in each of the columns there are more than one measurement systems, cm and inches or kg and lbs. This will all be converted to the european measurements: cm and kg.

- `Loan Date End`: it has a lot of "NULL". We have to check if it's missing values or strings with "NULL".

- `Value, Wage and Release Clause`: has "$" sign, "K" for thousands and "M" for million. We'll turn this into only numeric column.

- `W/F, SM and IR`: these features have stars after every number because it was extracted from the website of fifa. They will be removed.

Since we have so many columns, we will probably drop some if they have redundant information.

In [9]:
numeric_cols = [
    col_name
    for col_name, dtype in fifa.dtypes
    if dtype in ['int', 'bigint', 'double', 'float', 'long', 'decimal']
]

In [10]:
# in this phase, since all features are still categorized as strings, we will analyse all features, even non numeric
fifa.summary().show()

26/05/14 10:22:37 WARN DAGScheduler: Broadcasting large task binary with size 1164.6 KiB


+-------+------------------+-----------+--------------------+--------------------+--------------------+-----------+------------------+-----------------+------------------+--------------------+--------------------+---------+------+------+--------------+-----------------+-------------+-----------+-------------+-----+-----+--------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+-------

Since some players have "No Club", we can analyse inconsistencies in `Contract` or if they are interesting for our analysis or not. Also in name fields some players have special carachters. It may be because some players may be from different countries with letters that don't belong to our alphabet.

In [11]:
# show the distribution of values in "Value" for players who have "No Club"
fifa.filter(col("Club") == "No Club").select("Value", "Contract").distinct().show()

+-----+--------+
|Value|Contract|
+-----+--------+
|   €0|    Free|
+-----+--------+



All players without Club have no current value and contract is "Free" as expected

<a class="anchor" id="3_2_1">

### **3.3.1. Data Types**

[Back to TOC](#toc)
</a>

In [12]:
for col_name, dtype in df.dtypes:
    print(f"{col_name}: {dtype}")

ID: string
Name: string
LongName: string
photoUrl: string
playerUrl: string
Nationality: string
Age: string
↓OVA: string
POT: string
Club: string
Contract: string
Positions: string
Height: string
Weight: string
Preferred Foot: string
BOV: string
Best Position: string
Joined: string
Loan Date End: string
Value: string
Wage: string
Release Clause: string
Attacking: string
Crossing: string
Finishing: string
Heading Accuracy: string
Short Passing: string
Volleys: string
Skill: string
Dribbling: string
Curve: string
FK Accuracy: string
Long Passing: string
Ball Control: string
Movement: string
Acceleration: string
Sprint Speed: string
Agility: string
Reactions: string
Balance: string
Power: string
Shot Power: string
Jumping: string
Stamina: string
Strength: string
Long Shots: string
Mentality: string
Aggression: string
Interceptions: string
Positioning: string
Vision: string
Penalties: string
Composure: string
Defending: string
Marking: string
Standing Tackle: string
Sliding Tackle: string


All features are strings, even the statistic fields which are supposed to be numeric. This will be changed in the next steps to allow for proper analysis and visualization.

<a class="anchor" id="3_2_2">

### **3.3.2.  Check Missing Values**

[Back to TOC](#toc)
</a>

In [13]:
from pyspark.sql import functions as F

fifa.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in fifa.columns
]).show()

+---+----+--------+--------+---------+-----------+---+----+---+----+--------+---------+------+------+--------------+---+-------------+------+-------------+-----+----+--------------+---------+--------+---------+----------------+-------------+-------+-----+---------+-----+-----------+------------+------------+--------+------------+------------+-------+---------+-------+-----+----------+-------+-------+--------+----------+---------+----------+-------------+-----------+------+---------+---------+---------+-------+---------------+--------------+-----------+---------+-----------+----------+--------------+-----------+-----------+----------+---+---+---+---+---+---+---+---+---+---+---+----+
| ID|Name|LongName|photoUrl|playerUrl|Nationality|Age|↓OVA|POT|Club|Contract|Positions|Height|Weight|Preferred Foot|BOV|Best Position|Joined|Loan Date End|Value|Wage|Release Clause|Attacking|Crossing|Finishing|Heading Accuracy|Short Passing|Volleys|Skill|Dribbling|Curve|FK Accuracy|Long Passing|Ball Control|

In [14]:
missing_pct = fifa.select([
    (F.count(F.when(F.col(c).isNull(), c)) / F.count("*") * 100).alias(c)
    for c in fifa.columns
])

missing_pct.show(truncate=False)

+---+----+--------+--------+---------+-----------+---+----+---+----+--------+---------+------+------+--------------+---+-------------+------+----------------+-----+----+--------------+---------+--------+---------+----------------+-------------+-------+-----+---------+-----+-----------+------------+------------+--------+------------+------------+-------+---------+-------+-----+----------+-------+-------+--------+----------+---------+----------+-------------+-----------+------+---------+---------+---------+-------+---------------+--------------+-----------+---------+-----------+----------+--------------+-----------+-----------+----------+---+---+---+---+---+---+---+---+---+---+---+------------------+
|ID |Name|LongName|photoUrl|playerUrl|Nationality|Age|↓OVA|POT|Club|Contract|Positions|Height|Weight|Preferred Foot|BOV|Best Position|Joined|Loan Date End   |Value|Wage|Release Clause|Attacking|Crossing|Finishing|Heading Accuracy|Short Passing|Volleys|Skill|Dribbling|Curve|FK Accuracy|Long P

Here we confirm that `Loan Date End` and `Hits` have missing values, 17.966 and 2.595, respectively, which corresponds to 94.66% and 13.67%.

The number of missing values in `Loan Date End` is normal since, a player being loaned to another club isn't so common.

The missing values in `Hits` is also understandeble since it represents the number of visits to the player page in the plaform. Less known players may have never been visited :(

In [15]:
# check if there are cells with empty strings
empty_string_counts = fifa.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in fifa.columns
])
empty_string_counts.show(truncate=False)

+---+----+--------+--------+---------+-----------+---+----+---+----+--------+---------+------+------+--------------+---+-------------+------+-------------+-----+----+--------------+---------+--------+---------+----------------+-------------+-------+-----+---------+-----+-----------+------------+------------+--------+------------+------------+-------+---------+-------+-----+----------+-------+-------+--------+----------+---------+----------+-------------+-----------+------+---------+---------+---------+-------+---------------+--------------+-----------+---------+-----------+----------+--------------+-----------+-----------+----------+---+---+---+---+---+---+---+---+---+---+---+----+
|ID |Name|LongName|photoUrl|playerUrl|Nationality|Age|↓OVA|POT|Club|Contract|Positions|Height|Weight|Preferred Foot|BOV|Best Position|Joined|Loan Date End|Value|Wage|Release Clause|Attacking|Crossing|Finishing|Heading Accuracy|Short Passing|Volleys|Skill|Dribbling|Curve|FK Accuracy|Long Passing|Ball Control|

Change NULL strings to null values so they are detected in null count in case they exist.

<a class="anchor" id="3_2_3">

### **3.3.3.  Check for Duplicates**

[Back to TOC](#toc)
</a>

In [16]:
total_rows = fifa.count()
unique_rows = fifa.dropDuplicates().count()

print("Duplicated rows:", total_rows - unique_rows)

Duplicated rows: 0


In [17]:
fifa.groupBy("LongName") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------------------+-----+
|            LongName|count|
+--------------------+-----+
|       Michael Smith|    2|
|           Joe Riley|    2|
|      José Hernández|    2|
|     Richard Sánchez|    2|
|        Juan Ramírez|    2|
|          Diego Sosa|    2|
|        Jae Sung Lee|    2|
|       Luis Cárdenas|    2|
|     Alejandro Gómez|    2|
|            Shuai Li|    2|
|Matías de los Santos|    2|
|    Ignacio González|    2|
|  Sebastián Martínez|    2|
|        Adama Traoré|    3|
|        Riku Matsuda|    2|
|   Mathias Jørgensen|    2|
|     Leonardo Castro|    2|
|          Matt Smith|    2|
|         Tae Hee Lee|    2|
|         Greg Taylor|    2|
+--------------------+-----+
only showing top 20 rows


In [18]:
fifa.groupBy("LongName", "Age") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------------+---+-----+
|      LongName|Age|count|
+--------------+---+-----+
|José Hernández| 23|    2|
|     Peng Wang| 27|    2|
|  Adama Traoré| 25|    2|
|   Alan Medina| 22|    2|
|  Ryan Edwards| 26|    2|
|   Scott Brown| 35|    2|
|    Liam Kelly| 24|    2|
+--------------+---+-----+



In [19]:
fifa.groupBy("LongName", "Age", "Height", "Weight") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------+---+------+------+-----+
|LongName|Age|Height|Weight|count|
+--------+---+------+------+-----+
+--------+---+------+------+-----+



<a class="anchor" id="3_2_4">

### **3.3.4.  Check unique values**

[Back to TOC](#toc)
</a>

In [20]:
# check unique values in each column
for col_name in fifa.columns:
    unique_count = fifa.select(col_name).distinct().show()
    print(f"{col_name}: {unique_count} unique values")

+------+
|    ID|
+------+
|205069|
|193105|
|203605|
|234035|
|189280|
|205346|
|188942|
|247030|
|241854|
|222457|
|232709|
|212118|
|212214|
|238274|
|200206|
|188182|
|231436|
|207444|
|209039|
|140082|
+------+
only showing top 20 rows
ID: None unique values
+-----------------+
|             Name|
+-----------------+
|      M. Politano|
| A. Saint-Maximin|
|         G. Xhaka|
|          A. Izzo|
|          Postigo|
|    G. Schennikov|
|          F. Jara|
|        R. Assalé|
|         C. Adams|
|   J. Castelletto|
|        T. Almada|
|         C. Akpom|
|       F. Boulaya|
|         G. Akkan|
|Heriberto Tavares|
|     K. Nordfeldt|
|    B. Montenegro|
|    Rúben Freitas|
|   Flávio Rebeilo|
|        C. Cáceda|
+-----------------+
only showing top 20 rows
Name: None unique values
+--------------------+
|            LongName|
+--------------------+
|          Kenny Lala|
| Víctor Machín Pérez|
|          Bouna Sarr|
|        Lucas Castro|
|      Anwar El Ghazi|
|Rúben Afonso Borg...|

In [21]:
categorical_cols = [
    c for c, t in fifa.dtypes
    if t == "string"
]

for c in categorical_cols:
    print(f"\nValue counts for: {c}")

    fifa.groupBy(c) \
        .count() \
        .orderBy(F.desc("count")) \
        .show(10, truncate=False)


Value counts for: ID


+------+-----+
|ID    |count|
+------+-----+
|205069|1    |
|193105|1    |
|203605|1    |
|234035|1    |
|189280|1    |
|205346|1    |
|188942|1    |
|247030|1    |
|241854|1    |
|222457|1    |
+------+-----+
only showing top 10 rows

Value counts for: Name
+------------+-----+
|Name        |count|
+------------+-----+
|J. Rodríguez|13   |
|Paulinho    |8    |
|A. González |7    |
|J. García   |7    |
|R. Fernández|7    |
|M. Smith    |7    |
|J. Jones    |7    |
|J. González |7    |
|J. Williams |6    |
|J. Rojas    |6    |
+------------+-----+
only showing top 10 rows

Value counts for: LongName
+--------------------+-----+
|LongName            |count|
+--------------------+-----+
|Adama Traoré        |3    |
|Nicolás González    |3    |
|Danny Rose          |3    |
|Peng Wang           |3    |
|Diego Rodríguez     |3    |
|Liam Kelly          |3    |
|Michael Smith       |2    |
|Matías de los Santos|2    |
|Joe Riley           |2    |
|José Hernández      |2    |
+----------------

<a class="anchor" id="4">

# **4.  Data Preparation**

[Back to TOC](#toc)
</a>

<a class="anchor" id="4_1">

## **4.1.  Removing useless columns**

[Back to TOC](#toc)
</a>

Firstly, we will be dropping the columns `photoUrl` and `playerUrl` as it gives us no usefull information.

In [22]:
fifa = fifa.drop("Name","photoUrl", "playerUrl")

Here, after droping the column `Name`, we rename the column `LongName` to Name and also rename the `OVA` one.

In [23]:
fifa = fifa.withColumnRenamed("LongName", "Name")
fifa = fifa.withColumnRenamed("↓OVA", "OVA")

<a class="anchor" id="4_1">

## **4.2. Removing Special Characters and Treating Wrong Data Types**

[Back to TOC](#toc)
</a>

The `Club` column has invalid characters such as \n, here we will remove them.

In [24]:
fifa = fifa.withColumn("Club", trim(regexp_replace(col("Club"), r"[\n]", "")))

Here we will convert all the simple numeric features from string to integer.

In [29]:
stats_cols = [
    "Age", "OVA", "POT", "BOV", "Crossing", "Finishing", "Heading Accuracy", 
    "Short Passing", "Volleys", "Dribbling", "Curve", "FK Accuracy", "Long Passing", 
    "Ball Control", "Acceleration", "Sprint Speed", "Agility", "Reactions", "Balance", 
    "Shot Power", "Jumping", "Stamina", "Strength", "Long Shots", "Aggression", 
    "Interceptions", "Positioning", "Vision", "Penalties", "Composure", "Marking", 
    "Standing Tackle", "Sliding Tackle", "GK Diving", "GK Handling", "GK Kicking", 
    "GK Positioning", "GK Reflexes", "Total Stats", "Base Stats", "PAC", "SHO", 
    "PAS", "DRI", "DEF", "PHY", 
    "Attacking", "Skill", "Movement", "Power", "Mentality", "Defending", "Goalkeeping",
    "W/F", "SM", "IR"
]

In [30]:
for c in stats_cols:
    fifa = fifa.withColumn(c, regexp_replace(col(c), "[^0-9]", "").cast("int"))

In [31]:
for col_name, dtype in fifa.dtypes:
    print(f"{col_name}: {dtype}")

ID: string
Name: string
Nationality: string
Age: int
OVA: int
POT: int
Club: string
Contract: string
Positions: string
Height: string
Weight: string
Preferred Foot: string
BOV: int
Best Position: string
Joined: string
Loan Date End: string
Value: string
Wage: string
Release Clause: string
Attacking: int
Crossing: int
Finishing: int
Heading Accuracy: int
Short Passing: int
Volleys: int
Skill: int
Dribbling: int
Curve: int
FK Accuracy: int
Long Passing: int
Ball Control: int
Movement: int
Acceleration: int
Sprint Speed: int
Agility: int
Reactions: int
Balance: int
Power: int
Shot Power: int
Jumping: int
Stamina: int
Strength: int
Long Shots: int
Mentality: int
Aggression: int
Interceptions: int
Positioning: int
Vision: int
Penalties: int
Composure: int
Defending: int
Marking: int
Standing Tackle: int
Sliding Tackle: int
Goalkeeping: int
GK Diving: int
GK Handling: int
GK Kicking: int
GK Positioning: int
GK Reflexes: int
Total Stats: int
Base Stats: int
W/F: int
SM: int
A/W: string
D/W: s

In this step, we are normalizing all `Height` values to the metric system (cm) and converting the data type from string to integer.

In [32]:
fifa = fifa.withColumn("Height", 
    when(col("Height").contains("'"), 
        (split(col("Height"), "'")[0].cast("float") * 30.48) + 
        (regexp_replace(split(col("Height"), "'")[1], '"', "").cast("float") * 2.54)
    ).otherwise(regexp_replace(col("Height"), "cm", "").cast("float"))
).withColumn("Height", col("Height").cast("int"))

We will do the same for the `Weight` values.

In [33]:
fifa = fifa.withColumn("Weight", 
    when(col("Weight").contains("lbs"), 
        (regexp_replace(col("Weight"), "lbs", "").cast("float") * 0.453592)
    ).otherwise(regexp_replace(col("Weight"), "kg", "").cast("float"))
).withColumn("Weight", col("Weight").cast("int"))

Columns like `Value`, `Wage`, and `Hits` contain symbols ('€') and abbreviations ('M', 'K'). We apply a custom `parse_multipliers` function to clean the strings, calculate the final numerical values, and convert them to integers.

In [34]:
def parse_multipliers(column_name):
    return when(col(column_name).contains("M"), 
                (regexp_replace(col(column_name), "[€|M]", "").cast("float") * 1000000)) \
          .when(col(column_name).contains("K"), 
                (regexp_replace(col(column_name), "[€|K]", "").cast("float") * 1000)) \
          .otherwise(regexp_replace(col(column_name), "[€]", "").cast("float"))

for c in ["Value", "Wage", "Release Clause", "Hits"]:
    fifa = fifa.withColumn(c, parse_multipliers(c).cast("int"))

In [35]:
fifa.show(15)

+------+--------------------+-----------+---+---+---+-------------------+-----------+----------+------+------+--------------+---+-------------+------------+-------------+---------+------+--------------+---------+--------+---------+----------------+-------------+-------+-----+---------+-----+-----------+------------+------------+--------+------------+------------+-------+---------+-------+-----+----------+-------+-------+--------+----------+---------+----------+-------------+-----------+------+---------+---------+---------+-------+---------------+--------------+-----------+---------+-----------+----------+--------------+-----------+-----------+----------+---+---+------+------+---+---+---+---+---+---+---+----+
|    ID|                Name|Nationality|Age|OVA|POT|               Club|   Contract| Positions|Height|Weight|Preferred Foot|BOV|Best Position|      Joined|Loan Date End|    Value|  Wage|Release Clause|Attacking|Crossing|Finishing|Heading Accuracy|Short Passing|Volleys|Skill|Dribbl

In [63]:
# in this phase, since all features are still categorized as strings, we will analyse all features, even non numeric
fifa.summary().show()

26/05/14 11:09:03 WARN DAGScheduler: Broadcasting large task binary with size 1271.4 KiB


+-------+------------------+--------------------+-----------+------------------+-----------------+------------------+--------------------+--------------------+---------+------------------+-----------------+--------------+-----------------+-------------+-----------+-------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------

<a class="anchor" id="4_1">

## **4.3. Feature Engineering**

[Back to TOC](#toc)
</a>

A new feature engineered `N_Positions` to count the total playable positions per player, capturing their versatility.

In [38]:
fifa = fifa.withColumn("N_Positions", size(split(col("Positions"), ",")))

- **Contract**: In this feature, there are two dates separated by a "~" for each contract which we assume that it corresponds to the beggining and end date of every player in said club. We will separate the column in two, one for the year of entering the club and other for termination of contract.

- **Loan Date End**: it has a lot of "NULL". We have to check if it's missing values or strings with "NULL". CLUB, CONTRACT, PREFERRED FOOT, BEST POSITION, JOINED, LOAN DATE ENT, A/W, D/W, N_POSITIONS
- **COLOCAR DATAS EM DATE TYPE EM VEZ DE STRINGS?**


Since we have so many columns, we will probably drop some if they have redundant information.


**To do**: depois de passar para num voltar a fazer describe,

visiluaziaçoes das distribuiçoes


In [ ]:
a="Loan Date End"
fifa.filter((col(a) == "NULL")) \
    .select("Name", "Club", "Positions",a) \
    .show(20, truncate=False)

+----+----+---------+-------------+
|Name|Club|Positions|Loan Date End|
+----+----+---------+-------------+
+----+----+---------+-------------+



In [67]:
a="Loan Date End"
fifa.filter((col(a) == "")) \
    .select("Name", "Club", "Positions",a) \
    .show(20, truncate=False)

+----+----+---------+-------------+
|Name|Club|Positions|Loan Date End|
+----+----+---------+-------------+
+----+----+---------+-------------+



É NULL MESMO

In [69]:
a="Loan Date End"
fifa.filter(col(a).isNull()) \
    .select("Name", "Club", "Positions",a) \
    .show(20, truncate=False)

+---------------------------------+-------------------+----------+-------------+
|Name                             |Club               |Positions |Loan Date End|
+---------------------------------+-------------------+----------+-------------+
|Lionel Messi                     |FC Barcelona       |RW, ST, CF|NULL         |
|C. Ronaldo dos Santos Aveiro     |Juventus           |ST, LW    |NULL         |
|Jan Oblak                        |Atlético Madrid    |GK        |NULL         |
|Kevin De Bruyne                  |Manchester City    |CAM, CM   |NULL         |
|Neymar da Silva Santos Jr.       |Paris Saint-Germain|LW, CAM   |NULL         |
|Robert Lewandowski               |FC Bayern München  |ST        |NULL         |
|Mohamed Salah                    |Liverpool          |RW        |NULL         |
|Alisson Ramses Becker            |Liverpool          |GK        |NULL         |
|Kylian Mbappé                    |Paris Saint-Germain|ST, LW, RW|NULL         |
|Marc-André ter Stegen      